# Importing and vizualizing dataset

The first step in building an effective model for breast cancer detection is to import and preprocess the dataset. This section describes the methods used for loading the dataset, performing basic visualizations, and ensuring the data was in a suitable format for training the Convolutional Neural Network (CNN).

In [1]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import torchvision
import pandas as pd
import torchvision.transforms as transforms

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.utils.class_weight import compute_class_weight

from pathlib import Path

In [2]:
filedir = Path('./datasets/')

train_dataframe = pd.read_csv(filedir / 'train_data.csv')
test_dataframe = pd.read_csv(filedir / 'test_data.csv')

In [3]:
train_dataframe.head()

,ID,patient_id,breast density,left or right breast,image view,abnormality id,abnormality type,calc type,calc distribution,assessment,pathology,subtlety
0,Calc-Training_P_00005_RIGHT_CC_1,P_00005,3,RIGHT,CC,1,calcification,AMORPHOUS,CLUSTERED,3,MALIGNANT,3
1,Calc-Training_P_00005_RIGHT_MLO_1,P_00005,3,RIGHT,MLO,1,calcification,AMORPHOUS,CLUSTERED,3,MALIGNANT,3
2,Calc-Training_P_00007_LEFT_CC_1,P_00007,4,LEFT,CC,1,calcification,PLEOMORPHIC,LINEAR,4,BENIGN,4
3,Calc-Training_P_00007_LEFT_MLO_1,P_00007,4,LEFT,MLO,1,calcification,PLEOMORPHIC,LINEAR,4,BENIGN,4
4,Calc-Training_P_00008_LEFT_CC_1,P_00008,1,LEFT,CC,1,calcification,NaN,REGIONAL,2,BENIGN_WITHOUT_CALLBACK,3


In [4]:
train_dataframe.isnull().sum()

ID                        0
patient_id                0
breast density            0
left or right breast      0
image view                0
abnormality id            0
abnormality type          0
calc type                20
calc distribution       375
assessment                0
pathology                 0
subtlety                  0
dtype: int64

In [5]:
test_dataframe.isnull().sum()

ID                       0
patient_id               0
breast density           0
left or right breast     0
image view               0
abnormality id           0
abnormality type         0
calc type                4
calc distribution       63
assessment               0
pathology                0
subtlety                 0
dtype: int64

In [6]:
train_dataframe.dropna(inplace=True)
print(train_dataframe.shape)

cleaned_test = test_dataframe.dropna(inplace=True)
print(test_dataframe.shape)

(1149, 12)
(259, 12)


Considering train and test datasets contain missing values in `calc type` and `calc distribution` which could provide additional information for neural networks training it is good idea to drpp them since without medical expertise they are difficult to predict.

In [7]:
train_dataframe.head()

,ID,patient_id,breast density,left or right breast,image view,abnormality id,abnormality type,calc type,calc distribution,assessment,pathology,subtlety
0,Calc-Training_P_00005_RIGHT_CC_1,P_00005,3,RIGHT,CC,1,calcification,AMORPHOUS,CLUSTERED,3,MALIGNANT,3
1,Calc-Training_P_00005_RIGHT_MLO_1,P_00005,3,RIGHT,MLO,1,calcification,AMORPHOUS,CLUSTERED,3,MALIGNANT,3
2,Calc-Training_P_00007_LEFT_CC_1,P_00007,4,LEFT,CC,1,calcification,PLEOMORPHIC,LINEAR,4,BENIGN,4
3,Calc-Training_P_00007_LEFT_MLO_1,P_00007,4,LEFT,MLO,1,calcification,PLEOMORPHIC,LINEAR,4,BENIGN,4
20,Calc-Training_P_00010_LEFT_CC_1,P_00010,3,LEFT,CC,1,calcification,ROUND_AND_REGULAR-LUCENT_CENTER-DYSTROPHIC,DIFFUSELY_SCATTERED,2,BENIGN_WITHOUT_CALLBACK,4


Understanding the class distribution is essential for determining the suitability of additional feature encoding techniques. When certain classes are highly imbalanced, applying methods like one-hot encoding can lead to computational inefficiencies and inflated model complexity. By thoroughly analyzing the dataset’s characteristics, we can make informed decisions about preprocessing and encoding strategies, ensuring that the model remains computationally efficient and effective.

In [8]:
print(f"{'Column name':<25} Unique values")

for col in train_dataframe:
    print(f"{col:<25} {len(train_dataframe[col].unique())}")

Column name               Unique values
ID                        1149
patient_id                538
breast density            4
left or right breast      2
image view                2
abnormality id            6
abnormality type          1
calc type                 30
calc distribution         9
assessment                5
pathology                 3
subtlety                  5


# Data preprocessing

## 2.1. Dataset loader creation

Pytorch allows creation of custom datasets for our files which require implementation of these three functions: `__init__`, `__len__`, and `__getitem__`. For dataset loading and image manipulation openCV and Pillow libraries are used considering they offer vast majority of image enhancement/augmentation methods. We opted to use CLAHE since our dataset ontains grayscale images and additional contrast can be of a great help.

In [10]:
class BreastCancerDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None, apply_clahe=False):
        self.dataframe = dataframe
        self.image_dir = image_dir
        self.transform = transform
        self.apply_clahe = apply_clahe  # Flag to control CLAHE application
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image_path = f"{self.image_dir}/{row['ID']}.jpg"
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)  # Load as grayscale
        
        if image is None:
            raise FileNotFoundError(f"Image not found at path: {image_path}")
        
        if self.apply_clahe:
            clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
            image = clahe.apply(image)
        
        # Convert NumPy array to PIL image
        image = Image.fromarray(image)
        
        label = row['pathology']  # Target label
        label = 0 if label == 'BENIGN_WITHOUT_CALLBACK' else 1 if label == 'BENIGN' else 2  # Encode classes
        
        additional_features = row[['calc distribution']].astype(float)  # Additional features
        additional_features = torch.tensor(additional_features.values, dtype=torch.float32)
        
        if self.transform:
            image = self.transform(image)
        
        return image, label, additional_features

## 2.2. Image transformation and scaling

In order to make model more robust image manipulation is required. Such methods include image resizing and in our canse random horizontal flip alongside random rotation which rotates the image by an angle. Such feats yield more diverse images which can greatly improve neural network training.

In [11]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),  # Resize every image to 128 X 128 matrix
    transforms.RandomHorizontalFlip(),  # Make model more robust
    transforms.RandomRotation(10),
    transforms.ToTensor()  # Default scaling to [0, 1]
])

In [12]:
label_encoder = LabelEncoder()

train_dataframe['calc distribution'] = label_encoder.fit_transform(train_dataframe['calc distribution'])
test_dataframe['calc distribution'] = label_encoder.transform(test_dataframe['calc distribution'])

train_dataset = BreastCancerDataset(train_dataframe, filedir / 'train_cropped_images/', transform=transform, apply_clahe=True)
test_dataset = BreastCancerDataset(test_dataframe, filedir / 'test_cropped_images/', transform=transform, apply_clahe=True)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [13]:
train_dataframe.head()

,ID,patient_id,breast density,left or right breast,image view,abnormality id,abnormality type,calc type,calc distribution,assessment,pathology,subtlety
0,Calc-Training_P_00005_RIGHT_CC_1,P_00005,3,RIGHT,CC,1,calcification,AMORPHOUS,0,3,MALIGNANT,3
1,Calc-Training_P_00005_RIGHT_MLO_1,P_00005,3,RIGHT,MLO,1,calcification,AMORPHOUS,0,3,MALIGNANT,3
2,Calc-Training_P_00007_LEFT_CC_1,P_00007,4,LEFT,CC,1,calcification,PLEOMORPHIC,4,4,BENIGN,4
3,Calc-Training_P_00007_LEFT_MLO_1,P_00007,4,LEFT,MLO,1,calcification,PLEOMORPHIC,4,4,BENIGN,4
20,Calc-Training_P_00010_LEFT_CC_1,P_00010,3,LEFT,CC,1,calcification,ROUND_AND_REGULAR-LUCENT_CENTER-DYSTROPHIC,3,2,BENIGN_WITHOUT_CALLBACK,4


# Neural network model design

## 3.1. Convolutional Neural Networks

The decision to choose **CNN** as the primary classification model for breast cancer detection was driven by its ability to extract relevant features from medical images, its proven success in similar tasks, and its adaptability to large datasets. The use of CNNs allows for more accurate, reliable, and efficient detection, which is critical in providing timely and effective healthcare solutions.

In [18]:
class CNN(nn.Module):
    def __init__(self, num_classes, additional_features_size=0):
        super(CNN, self).__init__()
        # Convolutional blocks
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv_block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.dropout = nn.Dropout(0.5)
        
        # Fully connected layers
        self.fc1 = nn.Linear(128 * 16 * 16, 128)  # Adjust size, if input 64x64 change to 8*8, if input 128x128 change to 16*16
        self.relu = nn.ReLU()
        
        # Additional features
        self.additional_features_size = additional_features_size
        if self.additional_features_size > 0:
            self.fc_additional = nn.Sequential(
                nn.Linear(additional_features_size, 64),
                nn.ReLU(),
                nn.Linear(64, 64),
                nn.ReLU()
            )
        
        # Final classification layer
        self.fc_final = nn.Linear(128 + (64 if additional_features_size > 0 else 0), num_classes)
    
    def forward(self, x, additional_features=None):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.dropout(x)
        
        # Flatten
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        
        # Incorporate additional features
        if additional_features is not None and self.additional_features_size > 0:
            additional_features = self.fc_additional(additional_features)
            x = torch.cat((x, additional_features), dim=1)
        
        x = self.fc_final(x)
        return x

## 3.2. Hyperparameter definition

Choosing optimal hyperparameters is critical for achieving convergence and generalization in deep learning models. One of the most important hyperparameters in this study was the **learning rate**, as it directly influences the step size for gradient updates during optimization. We selected an initial learning rate of $10^{-4}$, with the option to adjust it dynamically using a **learning rate scheduler**. A higher learning rate (e.g., $10^{-4}$ or greater) was found to cause instability during training, with the loss oscillating significantly or even diverging in preliminary experiments.

To allow the model to adapt to varying optimization needs during training, a **learning rate scheduler** was employed. Specifically, the scheduler was configured to reduce the learning rate by a factor of 0.1 if the validation loss plateaued for a defined number of epochs. This dynamic adjustment helps prevent the model from stagnating in local minima while also minimizing the risk of overfitting.

In [15]:
num_epochs = 10  # Total training epochs
learning_rate = 1e-4  # Initial learning rate
weight_decay = 1e-3  # L2 regularization for optimizer

# Determine device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [19]:
# Model Parameters
num_classes = len(train_dataframe['pathology'].unique())  # Number of unique target classes
additional_features_size = 1  # Additional features for the model

# Model Initialization
model = CNN(num_classes, additional_features_size=additional_features_size).to(device)

# Optimizer and Scheduler
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.1,  # Reduce LR by a factor of 10
    patience=4   # Wait for 4 epochs of stagnation
)

# Class Weights for Handling Imbalance
def calculate_class_weights(dataset):
    """Calculate class weights for imbalanced datasets."""
    all_labels = [label for _, label, _ in dataset]  # Extract all labels from the dataset
    classes = np.unique(all_labels)  # Unique class labels
    class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=all_labels)
    return torch.tensor(class_weights, dtype=torch.float32).to(device)

class_weights_tensor = calculate_class_weights(train_dataset)
print(f"Class Weights: {class_weights_tensor}")

# Loss Function
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

Class Weights: tensor([4.5059, 0.7337, 0.7066], device='cuda:0')


## 3.3. Training loop

To efficiently train the Convolutional Neural Network (CNN) for breast cancer detection, we leveraged the power of GPU acceleration using **CUDA**, specifically utilizing the **NVIDIA GTX 1060** on my laptop. The use of CUDA allows faster convergence and improved model computation capabilities. In below training loop CUDA acceleration was incorporated by tagging images to GPU alongside model itself.

In [27]:
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    all_preds = []
    all_labels = []

    # Training loop
    for images, labels, additional_features in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        additional_features = additional_features.to(device)

        # Forward pass
        outputs = model(images, additional_features)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Accumulate loss and predictions
        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    train_loss /= len(train_loader)
    train_acc = accuracy_score(all_labels, all_preds)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}", end=", ")
    
    # Validation loop
    model.eval()
    val_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels, additional_features in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            additional_features = additional_features.to(device)
            
            outputs = model(images, additional_features)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss /= len(test_loader)
    val_acc = accuracy_score(all_labels, all_preds)
    scheduler.step(val_loss)
    print(f"Validation Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}")

Epoch 1/10, Loss: 0.5610, Accuracy: 0.6789, Validation Loss: 0.7697, Accuracy: 0.5792
Epoch 2/10, Loss: 0.5652, Accuracy: 0.6649, Validation Loss: 0.7760, Accuracy: 0.5985
Epoch 3/10, Loss: 0.5591, Accuracy: 0.6527, Validation Loss: 0.7630, Accuracy: 0.6062
Epoch 4/10, Loss: 0.5728, Accuracy: 0.6571, Validation Loss: 0.7478, Accuracy: 0.5985
Epoch 5/10, Loss: 0.5624, Accuracy: 0.6632, Validation Loss: 0.7492, Accuracy: 0.6139
Epoch 6/10, Loss: 0.5658, Accuracy: 0.6519, Validation Loss: 0.7728, Accuracy: 0.5869
Epoch 7/10, Loss: 0.5615, Accuracy: 0.6667, Validation Loss: 0.7539, Accuracy: 0.6062
Epoch 8/10, Loss: 0.5616, Accuracy: 0.6562, Validation Loss: 0.7794, Accuracy: 0.6100
Epoch 9/10, Loss: 0.5667, Accuracy: 0.6606, Validation Loss: 0.7394, Accuracy: 0.6062
Epoch 10/10, Loss: 0.5756, Accuracy: 0.6571, Validation Loss: 0.7786, Accuracy: 0.5907


In [23]:
save_path = Path("./60model.pth")
torch.save(model.state_dict(), save_path)

## 3.4. Using pretrained models

Considering custom made convolutional neural network model yields unsatisfactory results, can usage of pretrained models offer better performance? We opted for `ResNet` since it is implemented directly into Pytorch. Model is robust enough to provide success on our dataset.

In [28]:
resnet = torchvision.models.resnet18(weights="IMAGENET1K_V1")

resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, padding=1, bias=False)  # Modify first layer to accept greyscale images

num_ftrs = resnet.fc.in_features  # Get the number of input features for the final layer
resnet.fc = nn.Linear(num_ftrs, 3)  # Replace with a new layer for 3 classes

resnet.to(device)

ResNet(
  (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

Since we're performing fine tuning changing all parameters is not necesarry. For such purposes only fully connected layer requires tuning.

In [29]:
for param in resnet.parameters():
    param.requires_grad = False  # Freeze all layers

for param in resnet.fc.parameters():
    param.requires_grad = True  # Unfreeze the last fully connected layer

ResNet takes 224 x 224 input so scaling images to specific dimesions is required. Additionally, since ResNet only works with RGB images instead of greyscale ones, it was required to change input layer dimensionality and transformation normalization.

In [30]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),  # Make model more robust
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485], std=[0.229])  # ResNet normalization
])

train_dataset = BreastCancerDataset(train_dataframe, filedir / 'train_cropped_images/', transform=transform, apply_clahe=True)
test_dataset = BreastCancerDataset(test_dataframe, filedir / 'test_cropped_images/', transform=transform, apply_clahe=True)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [31]:
num_epochs = 10
optimizer = torch.optim.Adam(resnet.parameters(), lr=1e-4, weight_decay=1e-3)

If even a powerful model like ResNet is failes to surpass 50% accuracy, it strongly suggests that the issue might not lie in the model's architecture but rather in the dataset or its preprocessing.

In [34]:
for epoch in range(num_epochs):
    resnet.train()
    train_loss = 0
    all_preds = []
    all_labels = []

    # Training loop
    for images, labels, _ in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = resnet(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Accumulate loss and predictions
        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    train_loss /= len(train_loader)
    train_acc = accuracy_score(all_labels, all_preds)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}", end=", ")
    
    # Validation loop
    resnet.eval()
    val_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels, _ in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = resnet(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss /= len(test_loader)
    val_acc = accuracy_score(all_labels, all_preds)
    scheduler.step(val_loss)
    print(f"Validation Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}")

Epoch 1/10, Loss: 0.9470, Accuracy: 0.4961, Validation Loss: 0.9699, Accuracy: 0.4672
Epoch 2/10, Loss: 0.9538, Accuracy: 0.4883, Validation Loss: 1.0271, Accuracy: 0.5097
Epoch 3/10, Loss: 0.9350, Accuracy: 0.5022, Validation Loss: 0.9587, Accuracy: 0.4788
Epoch 4/10, Loss: 0.9327, Accuracy: 0.5048, Validation Loss: 1.0072, Accuracy: 0.4093
Epoch 5/10, Loss: 0.9179, Accuracy: 0.5379, Validation Loss: 0.9335, Accuracy: 0.4903
Epoch 6/10, Loss: 0.9299, Accuracy: 0.5091, Validation Loss: 0.9379, Accuracy: 0.4247
Epoch 7/10, Loss: 0.9259, Accuracy: 0.4839, Validation Loss: 0.9192, Accuracy: 0.5483
Epoch 8/10, Loss: 0.9281, Accuracy: 0.5161, Validation Loss: 0.9805, Accuracy: 0.4286
Epoch 9/10, Loss: 0.9167, Accuracy: 0.5178, Validation Loss: 0.9999, Accuracy: 0.4517
Epoch 10/10, Loss: 0.9243, Accuracy: 0.4900, Validation Loss: 0.9780, Accuracy: 0.5019


## 3.5. Last breath of hope, simple CNN model

Considering both previous examples gave approximate result, where our model reigned supreme with 0.1 better test set accuracy, it is safe to say that complicated models tend to overfitt the data. Such issue can be solved by reducing complexity of models. Last hope is simple CNN with 4 convolutional layers and input image dimensions of 32 x 32.

In [35]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.conv_layer1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3)
        self.conv_layer2 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3)
        self.max_pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv_layer3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)
        self.conv_layer4 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3)
        self.max_pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.fc1 = nn.Linear(1600, 128)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)
    
    # Progresses data across layers    
    def forward(self, x):
        x = self.conv_layer1(x)
        x = self.conv_layer2(x)
        x = self.max_pool1(x)
        
        x = self.conv_layer3(x)
        x = self.conv_layer4(x)
        x = self.max_pool2(x)
                
        x = x.reshape(x.size(0), -1)
        
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        return x

In [36]:
num_epochs = 20

simple_model = SimpleCNN(3)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(simple_model.parameters(), lr=1e-4, weight_decay=0.005)

simple_model.to(device)

SimpleCNN(
  (conv_layer1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
  (conv_layer2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1))
  (max_pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv_layer3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (conv_layer4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
  (max_pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=1600, out_features=128, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=128, out_features=3, bias=True)
)

In [37]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize every image to 32x32 matrix
    transforms.ToTensor()  # Default scaling to [0, 1]
])

train_dataset = BreastCancerDataset(train_dataframe, filedir / 'train_cropped_images/', transform=transform, apply_clahe=True)
test_dataset = BreastCancerDataset(test_dataframe, filedir / 'test_cropped_images/', transform=transform, apply_clahe=True)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [39]:
for epoch in range(num_epochs):
    train_loss = 0
    all_preds = []
    all_labels = []

    # Training loop
    for images, labels, _ in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = simple_model(images)
        loss = criterion(outputs, labels)
        
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Accumulate loss and predictions
        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    train_loss /= len(train_loader)
    train_acc = accuracy_score(all_labels, all_preds)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}", end=", ") 

    # Validation loop
    val_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels, _ in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = simple_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss /= len(test_loader)
    val_acc = accuracy_score(all_labels, all_preds)
    print(f"Validation Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}")

Epoch 1/20, Loss: 0.8689, Accuracy: 0.5170, Validation Loss: 0.7544, Accuracy: 0.5212
Epoch 2/20, Loss: 0.8680, Accuracy: 0.5370, Validation Loss: 0.7763, Accuracy: 0.5598
Epoch 3/20, Loss: 0.8626, Accuracy: 0.5553, Validation Loss: 0.7824, Accuracy: 0.5714
Epoch 4/20, Loss: 0.8635, Accuracy: 0.5352, Validation Loss: 0.7571, Accuracy: 0.5251
Epoch 5/20, Loss: 0.8544, Accuracy: 0.5448, Validation Loss: 0.7677, Accuracy: 0.5676
Epoch 6/20, Loss: 0.8492, Accuracy: 0.5553, Validation Loss: 0.7453, Accuracy: 0.5521
Epoch 7/20, Loss: 0.8483, Accuracy: 0.5431, Validation Loss: 0.7603, Accuracy: 0.5598
Epoch 8/20, Loss: 0.8407, Accuracy: 0.5657, Validation Loss: 0.7786, Accuracy: 0.5367
Epoch 9/20, Loss: 0.8401, Accuracy: 0.5727, Validation Loss: 0.7437, Accuracy: 0.5521
Epoch 10/20, Loss: 0.8340, Accuracy: 0.5744, Validation Loss: 0.7744, Accuracy: 0.5444
Epoch 11/20, Loss: 0.8316, Accuracy: 0.5753, Validation Loss: 0.7883, Accuracy: 0.5328
Epoch 12/20, Loss: 0.8399, Accuracy: 0.5701, Validat

As seen above even simple model produces insufficient results regarding test and train accuracy. This can be a strong indicator of invalid dataset, whether it is too small in size or some labels are wrong.